In [1]:
#Importing all the required libraries

# Document loading & chunking
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings #Only install a class from huggingface

# Vector store
from langchain_community.vectorstores import Chroma

# LLM
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
import os

C:\Users\manog\AppData\Local\Temp\ipykernel_54856\995767986.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
#Loading the book
documents_map = {
    "DIFC_Court_Rules.pdf": "DIFC Court Rules",
    "DFSA_Mkt_Rules_24-25.pdf": "DFSA Markets Rules",
    "DFSA_Gen_module.pdf": "DFSA General Module",
    "DFSA_Gen_Appendix_1.pdf": "DFSA GEN Appendix 1",
    "DFSA_Gen_Appendix_2.pdf": "DFSA GEN Appendix 2",
    "DFSA_Gen_Appendix_3.pdf": "DFSA GEN Appendix 3",
    "DFSA_Gen_Appendix_4.pdf": "DFSA GEN Appendix 4",
    "UAE_Federal_Aml LAW.pdf": "UAE Federal AML Law",
    "DIFC_Data_Protection_Law.pdf": "DIFC Data Protection Law",
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\nArticle", "\nSection", "\n\n", "\n"]
)

all_chunks = []

for filename, doc_name in documents_map.items():
    print(f"Loading {doc_name}...")
    loader = PyPDFLoader(filename)
    pages = loader.load()
    chunks = splitter.split_documents(pages)
    
    # Tag every chunk with which document it came from
    for chunk in chunks:
        chunk.metadata["source_doc"] = doc_name
    
    all_chunks.extend(chunks)
    print(f"  → {len(chunks)} chunks")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

Loading DIFC Court Rules...


MemoryError: 

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="./difc_chroma_db"
)

print("All documents embedded and stored.")

C:\Users\manog\AppData\Local\Temp\ipykernel_8336\4161790304.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\manog\OneDrive\Desktop\Summer\Compliance RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manog\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Done! Embeddings stored in ChromaDB


In [ ]:
# Load existing vectorstore (no re-embedding needed)
vectorstore = Chroma(
    persist_directory="./difc_chroma_db",
    embedding_function=embeddings
)

# Set up Groq
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model_name="llama-3.1-8b-instant"
)

# Your question
query = "What happens if a defendant fails to respond to a claim?"

# Retrieve relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
relevant_chunks = retriever.invoke(query)

# Build prompt
context = "\n\n".join([
    f"[Page {doc.metadata.get('page', 'N/A')}]: {doc.page_content}"
    for doc in relevant_chunks
])

prompt = f"""You are a compliance assistant for DIFC regulations. 
Answer the question using ONLY the context below.
Cite the page number for every point you make.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {query}
"""

response = llm.invoke(prompt)
print(response.content)

If a defendant fails to respond to a claim, the claimant can seek default judgment against the defendant. 

To obtain default judgment, the claimant must ensure that the claim is one that the Court has power to hear and decide (Page 13.23). 

Additionally, if the claim was served outside the jurisdiction and the defendant has not acknowledged service, the evidence must establish that the claim is one that the Court has power to hear and decide (Page 13.23) and that the claim form is served out of the jurisdiction (Page 77).

If the defendant has not returned an admission to the claimant under Rule 15.14, the claimant can proceed with seeking default judgment (Page 15.10).

However, the Court may proceed with the trial in the absence of the defendant, but with the following consequences:

- If the defendant does not attend, the Court may strike out the defendant's defence and allow the claimant to prove any counterclaim and obtain judgment on the counterclaim and for costs (Page 35.14(2